# 건물 특성과 에너지 효율성의 관계 분석

**분석 대상:** UCI Energy Efficiency (768개 건물, X1-X8, Y1-Y2)  
**모델링 목표:** Separate Models 방식으로 난방 부하(Y1)와 냉방 부하(Y2)를 각각 설명·예측  
**학습 범위:** 기초통계량, 분포·왜도/첨도, Pearson 상관관계, `LinearRegression`, 잔차·Q-Q plot

> 이 노트북은 수업 자료의 분석 흐름을 따라 숫자 → 현상 → 도메인 의미로 해석한다.
> ANOVA, VIF, Tukey HSD, 하이퍼파라미터 최적화는 사용하지 않는다.

In [ ]:
import io
import urllib.request
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_style("whitegrid")

# macOS에서 Matplotlib 한글 깨짐 방지
KOREAN_FONT_PATH = "/System/Library/Fonts/Supplemental/AppleGothic.ttf"
font_manager.fontManager.addfont(KOREAN_FONT_PATH)
KOREAN_FONT = font_manager.FontProperties(fname=KOREAN_FONT_PATH).get_name()
plt.rcParams["font.family"] = KOREAN_FONT
plt.rcParams["axes.unicode_minus"] = False

COLORS = {"Y1": "#E76F51", "Y2": "#2A9D8F"}
NAMES = {
    "X1": "Relative Compactness", "X2": "Surface Area",
    "X3": "Wall Area", "X4": "Roof Area", "X5": "Overall Height",
    "X6": "Orientation", "X7": "Glazing Area",
    "X8": "Glazing Area Distribution", "Y1": "Heating Load",
    "Y2": "Cooling Load"
}

## 1. 데이터 로딩과 품질 확인

UCI 공식 URL에서 Energy Efficiency 원본 ZIP 파일을 내려받아 메모리에서 직접 읽는다.
따라서 별도의 로컬 데이터 경로 없이 다른 PC나 Colab에서도 같은 원본으로 실행할 수 있다.

In [ ]:
import zipfile

DATA_URL = "https://archive.ics.uci.edu/static/public/242/energy+efficiency.zip"
raw = urllib.request.urlopen(DATA_URL).read()

with zipfile.ZipFile(io.BytesIO(raw)) as z:
    with z.open("ENB2012_data.xlsx") as f:
        df = pd.read_excel(f)

df.columns = [f"X{i}" for i in range(1, 9)] + ["Y1", "Y2"]
print("data source:", DATA_URL)
print("shape:", df.shape)
display(df.head())
print("\nmissing values:")
display(df.isnull().sum().to_frame("missing"))

**해석**

- 768행, 설명변수 8개, 목표변수 2개이며 결측치는 없다.
- X6(방향)과 X8(유리창 면적 분포)은 숫자로 기록되어 있지만 순서 자체가 연속적인 물리량은 아니다.
  따라서 Pearson 상관계수만으로 중요도를 판단하지 않고 그룹별 분포와 변수의 의미를 함께 본다.

## 분석 학습 기록: 어떤 순서로 판단할 것인가?

처음에는 목표변수와 상관계수가 큰 변수만 고르면 된다고 생각할 수 있다. 하지만 상관계수는 관계의 한 단면일 뿐이고,
변수의 분포가 치우쳤는지, 값이 얼마나 퍼져 있는지, 다른 설명변수와 같은 정보를 반복하는지도 함께 봐야 한다.

따라서 아래 순서를 분석의 판단 기준으로 삼았다.

In [ ]:
from matplotlib.patches import FancyBboxPatch

steps = [
    ("1. 질문", "어떤 건물 특성이\n에너지 부하와 관련될까?"),
    ("2. 중심·퍼짐", "평균·중앙값·표준편차로\n대표값과 변동 확인"),
    ("3. 분포·대칭", "왜도·첨도로\n치우침과 꼬리 확인"),
    ("4. 관계", "산점도·상자그림·상관으로\n변수 간 동행 확인"),
    ("5. 해석·선정", "건물 의미와 중복성을 더해\n최종 변수 결정"),
    ("6. 모델", "LinearRegression의\n설명력·오차·잔차 확인"),
]

fig, ax = plt.subplots(figsize=(16, 3.2))
ax.set_xlim(0, 16)
ax.set_ylim(0, 3)
ax.axis("off")

for i, (title, detail) in enumerate(steps):
    x = 0.25 + i * 2.63
    box = FancyBboxPatch(
        (x, 0.65), 2.15, 1.65,
        boxstyle="round,pad=0.08,rounding_size=0.12",
        facecolor="#EAF4F8", edgecolor="#2A6F97", linewidth=1.5
    )
    ax.add_patch(box)
    ax.text(x + 1.075, 1.78, title, ha="center", va="center",
            fontsize=11, fontweight="bold", color="#17324D")
    ax.text(x + 1.075, 1.18, detail, ha="center", va="center",
            fontsize=9, color="#37474F")
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + 2.52, 1.48), xytext=(x + 2.18, 1.48),
                    arrowprops=dict(arrowstyle="->", color="#E76F51", lw=1.8))

plt.title("EDA에서 모델 개발까지의 학습 흐름", fontsize=15, pad=8)
plt.show()

**학습 포인트**

- 평균과 중앙값의 차이는 분포의 치우침을 의심하게 한다.
- 표준편차는 값의 단위 영향을 받으므로 서로 단위가 다른 변수끼리 크기만 직접 비교하지 않는다.
- 왜도·첨도는 “좋고 나쁨”이 아니라 평균·표준편차만으로 분포를 요약해도 되는지 판단하는 보조정보다.
- 상관관계가 크더라도 다른 설명변수와 거의 같은 정보를 담으면 모두 모델에 넣지 않는다.
- 마지막 변수 선택은 통계량에 건물 설계 의미를 더한 판단이어야 한다.

## 2. 기초통계량: 중심, 퍼짐, 분포

In [ ]:
basic_stats = pd.DataFrame({
    "mean": df.mean(),
    "median": df.median(),
    "std": df.std(),
    "min": df.min(),
    "max": df.max(),
    "skew": df.skew(),
    "kurtosis": df.kurt(),
})
display(basic_stats.round(3))

target_stats = df[["Y1", "Y2"]].agg(
    ["mean", "median", "std", "skew", "kurt", "min", "max"]
)
print("Y1·Y2 기술통계")
display(target_stats.round(3))

### 학습 기록 1: 중심과 퍼짐을 어떻게 읽었는가?

| 변수 | 확인한 숫자 | 숫자에서 읽은 현상 | 다음 판단 |
|---|---:|---|---|
| X1 상대적 집약도 | 평균 0.764, 중앙값 0.750, 표준편차 0.106 | 평균과 중앙값이 비슷하고 범위가 좁다 | 작은 수치 차이라도 다른 형상 변수와 함께 확인 |
| X2 표면적 | 평균 671.71, 중앙값 673.75, 표준편차 88.09 | 중심이 거의 같아 대칭에 가깝고 건물 규모 차이가 충분히 존재 | 규모 대표 후보로 유지 |
| X3 벽 면적 | 평균·중앙값 318.50, 표준편차 43.63 | 중심이 일치하지만 왜도 0.53으로 일부 큰 벽 면적 설계가 존재 | 구조 정보 후보로 유지 |
| X5 전체 높이 | 평균·중앙값 5.25, 표준편차 1.75 | 3.5와 7.0의 두 설계 수준으로 나뉨 | 연속적 직선관계보다 그룹 차이도 확인 |
| X6 방향 | 코드 2·3·4·5, 각 192개 | 크기를 뜻하지 않는 범주 코드 | 그룹별 Y1·Y2 중심과 분포를 별도 비교 |
| X7 유리창 면적 | 평균 0.234, 중앙값 0.250, 표준편차 0.133 | 0~0.4 범위에서 창호 조건이 넓게 변함 | 외피 개구부 후보로 유지 |
| X8 유리창 분포 | 코드 0·1·2·3·4·5 | 숫자 간 거리가 물리적 크기를 뜻하지 않는 범주 코드 | 최종 선형회귀 입력에서 제외 |
| Y1 난방 부하 | 평균 22.31 > 중앙값 18.95, 표준편차 10.09 | 높은 부하 설계가 평균을 끌어올리며 건물 간 차이가 큼 | 분포와 구조 변수 관계를 시각화 |
| Y2 냉방 부하 | 평균 24.59 > 중앙값 22.08, 표준편차 9.51 | 높은 부하 관측값이 오른쪽에 더 분포 | Y1과 같은 수준으로 분포·관계 확인 |

이 단계에서 “표준편차가 큰 X2가 가장 중요하다”고 결론 내리지 않았다. X2는 단위 자체가 크기 때문이다.
중요도는 목표변수와의 관계 및 설명변수 중복까지 확인한 뒤 결정한다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, ax, color in [
    ("Y1", axes[0, 0], COLORS["Y1"]),
    ("Y2", axes[0, 1], COLORS["Y2"]),
]:
    sns.histplot(df[col], kde=True, color=color, ax=ax)
    ax.axvline(df[col].mean(), color="black", ls="--",
               label=f"mean={df[col].mean():.2f}")
    ax.axvline(df[col].median(), color="#457B9D", ls=":",
               label=f"median={df[col].median():.2f}")
    ax.set_title(f"{col} histogram")
    ax.set_xlabel(f"{col} unit")
    ax.legend()

sns.boxplot(y=df["Y1"], color=COLORS["Y1"], ax=axes[1, 0])
axes[1, 0].set_title("Y1 boxplot")
axes[1, 0].set_ylabel("Y1 unit")
sns.boxplot(y=df["Y2"], color=COLORS["Y2"], ax=axes[1, 1])
axes[1, 1].set_title("Y2 boxplot")
axes[1, 1].set_ylabel("Y2 unit")
plt.tight_layout()
plt.show()

**해석**

- Y1은 평균 **22.31**, 중앙값 **18.95**, 표준편차 **10.09**, 왜도 **0.36**, 초과첨도 **-1.25**,
  최소 **6.01**, 최대 **43.10**이다.
- Y2는 평균 **24.59**, 중앙값 **22.08**, 표준편차 **9.51**, 왜도 **0.40**, 초과첨도 **-1.15**,
  최소 **10.90**, 최대 **48.03**이다.
- 두 목표 모두 평균이 중앙값보다 크고 왜도가 양수이므로 높은 부하 값이 오른쪽에 더 분포한다.
- pandas의 `kurt()`는 정규분포를 0으로 두는 **초과첨도**다. 두 값이 음수라는 사실만으로 봉우리 모양을
  단정하지 않고, 히스토그램에서 여러 설계군이 섞인 형태와 함께 해석한다.
- 데이터 출처에 Y1·Y2의 물리 단위가 명시되어 있지 않으므로 kWh 같은 단위를 임의로 붙이지 않고
  “Y1 단위”, “Y2 단위”로 표현한다.

### 학습 기록 2: 처음 생각을 어떻게 수정했는가?

- **처음 생각:** 평균과 표준편차만 보면 Y1·Y2의 일반적인 크기와 변동을 충분히 설명할 수 있을 것이다.
- **확인 결과:** 두 목표 모두 평균이 중앙값보다 높고 히스토그램에는 여러 설계군이 섞인 형태가 보였다.
- **수정한 해석:** 하나의 정규분포처럼 요약하기보다 건물 높이·형상 그룹이 섞인 분포로 보아야 한다.
- **다음 행동:** 주요 설명변수와 Y1·Y2 관계, X6 그룹별 분포를 각각 확인했다.

## 3. 변수 관계: 숫자에서 건물 현상으로

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Pearson correlation matrix")
plt.tight_layout()
plt.show()

correlation_table = corr[["Y1", "Y2"]]
print("각 변수와 Y1·Y2의 Pearson 상관계수")
display(correlation_table.round(3))

y1_y2_corr = corr.loc["Y1", "Y2"]
print(f"Y1-Y2 Pearson correlation: {y1_y2_corr:.6f}")

**핵심 관계 해석**

1. **전체 높이(X5)**는 Y1과 **0.889**, Y2와 **0.896**의 비교적 강한 양의 선형관계를 보인다.
   이 데이터에서는 X5가 큰 설계군에서 Y1과 Y2가 상대적으로 높은 경향이 관찰되었다. 다만 상관관계만으로
   X5가 부하 증가의 직접적인 원인이라고 단정할 수는 없다.
2. **지붕 면적(X4)**은 Y1과 **-0.862**, Y2와 **-0.863**이다. 또한 X4-X5는 **-0.973**이므로,
   X4의 관계를 지붕 면적만의 독립적 효과가 아니라 서로 결합된 건물 형상 관계로 해석했다.
3. **표면적(X2)**은 Y1과 **-0.658**, Y2와 **-0.673**, **벽 면적(X3)**은 각각 **0.456**, **0.427**이다.
   두 변수는 건물 외피 규모와 벽 면적이라는 서로 다른 구조 정보를 나타낸다.
4. **유리창 면적(X7)**은 Y1과 **0.270**, Y2와 **0.208**로 형상 변수보다 약하지만 별도의 창호 면적 특성을
   나타낸다. 이는 창호 면적과 부하가 함께 변하는 경향이며 열손실 같은 원인을 확정하는 결과는 아니다.
5. **Y1-Y2 상관계수는 0.975862**이다. 같은 설계에서 두 부하가 강하게 함께 변하지만, Separate Models에서는
   한 목표변수를 다른 모델의 입력으로 사용하지 않는다.

**범주형 코드 주의:** X6과 X8도 숫자로 저장되어 상관표에 계산값이 표시되지만, 숫자 간 차이가 연속적인
물리량을 뜻하지 않는다. 따라서 두 변수의 Pearson 상관계수만으로 중요도를 판단하거나 선형 효과를 해석하지 않는다.

### 학습 기록 3: 상관계수만으로 고르지 않은 이유

- X4는 Y1과의 절댓값 상관이 0.862로 매우 크다. 처음에는 반드시 선택해야 할 변수처럼 보였다.
- 그러나 X4-X5=-0.973을 확인하니, 큰 지붕 면적은 낮은 건물 설계군과 거의 묶여 있었다.
- 따라서 X4의 큰 상관을 “지붕 면적의 독립적 효과”라고 말할 수 없었다.
- 같은 형상 정보를 반복하는 X1·X4 대신 규모 대표 X2와 높이 X5를 남기는 쪽이 해석하기 쉽다고 판단했다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.scatterplot(data=df, x="X5", y="Y1", alpha=.4, color=COLORS["Y1"], ax=axes[0, 0])
axes[0, 0].set_title("X5 vs Y1")
sns.scatterplot(data=df, x="X5", y="Y2", alpha=.4, color=COLORS["Y2"], ax=axes[0, 1])
axes[0, 1].set_title("X5 vs Y2")
sns.scatterplot(data=df, x="X7", y="Y1", alpha=.4, color=COLORS["Y1"], ax=axes[1, 0])
axes[1, 0].set_title("X7 vs Y1")
sns.scatterplot(data=df, x="X7", y="Y2", alpha=.4, color=COLORS["Y2"], ax=axes[1, 1])
axes[1, 1].set_title("X7 vs Y2")
plt.tight_layout()
plt.show()

**그래프가 의미 있는 이유**

- X5 산점도에서 두 높이 수준에 따른 Y1·Y2 위치 차이가 모두 관찰된다. 다만 다른 형상 변수와 함께 설계되었으므로
  높이의 직접적인 인과효과로 단정하지 않는다.
- X7 산점도에서는 같은 창호 면적에서도 Y1·Y2 변동이 커서 X7 하나만으로 부하를 설명할 수 없음을 확인했다.

### X6 방향 코드: 그룹별 중심과 분포

In [ ]:
x6_group_stats = df.groupby("X6")[["Y1", "Y2"]].agg(
    ["mean", "median", "std", "count"]
)
display(x6_group_stats.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=df, x="X6", y="Y1", color=COLORS["Y1"], ax=axes[0])
axes[0].set_title("Y1 distribution by X6 orientation group")
axes[0].set_xlabel("X6 orientation code")
axes[0].set_ylabel("Y1 unit")
sns.boxplot(data=df, x="X6", y="Y2", color=COLORS["Y2"], ax=axes[1])
axes[1].set_title("Y2 distribution by X6 orientation group")
axes[1].set_xlabel("X6 orientation code")
axes[1].set_ylabel("Y2 unit")
plt.tight_layout()
plt.show()

X6은 방향을 구분하는 범주형 코드이므로 코드 간 수치 차이에 연속적인 선형 의미를 부여하지 않았다.
실제 그룹 통계에서 Y1 평균은 **22.260~22.381**, Y2 평균은 **24.313~24.954** 범위였고 각 그룹은
모두 **192개**였다. 상자그림에서도 분포가 크게 겹친다. 이 데이터에서는 네 방향 그룹 사이의 중심 차이가
각 그룹 내부 표준편차(Y1 약 10.1, Y2 약 9.3~9.8)에 비해 작게 관찰되었다. 이는 현재 데이터에 대한
기술적 비교이며 방향의 일반적인 효과가 없다는 인과적 결론은 아니다.

## 4. EDA 기반 변수 선택

In [ ]:
geometry_corr = corr.loc[["X1", "X2", "X4", "X5"], ["X1", "X2", "X4", "X5"]]
display(geometry_corr.round(3))

상관계수만 큰 변수를 고르지 않고, 중복성과 건물 의미를 함께 고려한다.

- **X1, X2, X4, X5는 강하게 중복**된다. 예: X1-X2=-0.992, X4-X5=-0.973.
- 네 변수를 모두 넣으면 같은 “건물 형상/규모” 정보를 여러 번 세게 반영해 개별 계수 해석이 불안정해진다.
- **X2**는 Y1·Y2와 각각 -0.658, -0.673의 관계를 보이며 건물 외피 규모를 나타내는 대표 변수로 선택했다.
- **X3**은 Y1·Y2와 각각 0.456, 0.427이며 벽 면적이라는 별도의 구조 특성을 나타내므로 선택했다.
- **X5**는 Y1·Y2와 각각 0.889, 0.896이며 건물 높이 특성을 대표하므로 선택했다.
- **X7**은 Y1·Y2 상관이 0.270, 0.208로 상대적으로 약하지만 창호 면적이라는 별도의 건물 특성을 반영하기 위해 선택했다.
- **X6**은 방향 범주 코드이므로 그룹별 통계와 분포로 분석했으며 최종 선형회귀 입력에서는 제외했다.
- **X8은 유리창 분포 유형을 나타내는 코드형 범주변수이므로 숫자 간 차이를 연속적인 물리량으로 해석하기 어렵다.
  따라서 Pearson 상관계수만으로 중요도를 판단하지 않았으며 최종 선형회귀 입력변수에서는 제외하였다.**

**최종 선정 판단표**

| 변수 | 통계량 근거 | 도메인 해석 | 결정 |
|---|---|---|---|
| X2 | Y1 -0.658, Y2 -0.673; X1-X2 -0.992 | 외피 규모 대표 | 선택 |
| X3 | Y1 0.456, Y2 0.427 | 벽 면적 특성 | 선택 |
| X5 | Y1 0.889, Y2 0.896; X4-X5 -0.973 | 높이 특성 대표 | 선택 |
| X7 | Y1 0.270, Y2 0.208 | 별도의 창호 면적 특성 | 선택 |
| X1 | X2와 강한 중복 | X2로 규모 정보 대표 | 제외 |
| X4 | X5와 강한 중복 | X5로 높이·형상 정보 대표 | 제외 |
| X6 | 방향 그룹별 중심·분포 비교 | 연속량이 아닌 방향 코드 | 제외 |
| X8 | 코드 숫자 간 물리적 간격 없음 | 연속량이 아닌 유리창 분포 유형 | 제외 |

최종 공통 설명변수는 **X2, X3, X5, X7**이다. 상관계수가 높은 순서만으로 고르지 않고,
Y1·Y2와의 관계, 설명변수 간 중복, 건물 특성의 의미와 코드형 변수의 해석 한계를 함께 고려했다.

## 5. 선형회귀 모델 개발: Separate Models

과제의 두 방안 중 **Separate Models**를 선택한다. 같은 최종 설명변수를 사용하되 Y1과 Y2를 각각 별도의
`LinearRegression`으로 학습한다. 두 목표변수의 상관관계를 두 번째 모델 입력으로 사용하지 않으므로,
각 모델의 결과를 독립적으로 해석할 수 있다. 별도 최적화는 진행하지 않는다.

Y1과 Y2는 서로 다른 목표변수이므로 각각 독립적인 선형회귀 모델로 학습하였다. 두 목표변수는 동일한 건물 형상 및
창호 조건에서 측정된 부하이므로 두 모델의 결과를 동일한 기준에서 비교하기 위해 공통 설명변수를 사용하였다.
각 모델의 회귀계수와 성능은 목표변수별로 독립적으로 해석하였다.

In [ ]:
selected = ["X2", "X3", "X5", "X7"]
X = df[selected]
X_train, X_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    X, df["Y1"], df["Y2"], test_size=0.2, random_state=42
)

model_y1 = LinearRegression()
model_y2 = LinearRegression()
model_y1.fit(X_train, y1_train)
model_y2.fit(X_train, y2_train)

assert model_y1 is not model_y2
assert list(X.columns) == ["X2", "X3", "X5", "X7"]
assert y1_train.index.equals(y2_train.index)
assert y1_test.index.equals(y2_test.index)
print("별도 모델 객체:", model_y1 is not model_y2)
print("공통 학습 행 일치:", y1_train.index.equals(y2_train.index))
print("공통 테스트 행 일치:", y1_test.index.equals(y2_test.index))

y1_train_pred = model_y1.predict(X_train)
y1_test_pred = model_y1.predict(X_test)
y2_train_pred = model_y2.predict(X_train)
y2_test_pred = model_y2.predict(X_test)

coef_table = pd.DataFrame({
    "feature": selected,
    "Y1 coefficient": model_y1.coef_,
    "Y2 coefficient": model_y2.coef_,
})
print(f"Y1 intercept: {model_y1.intercept_:.4f}")
print(f"Y2 intercept: {model_y2.intercept_:.4f}")
display(coef_table.round(4))

metrics = pd.DataFrame({
    "모델": ["Y1 모델", "Y2 모델"],
    "Train R2": [
        r2_score(y1_train, y1_train_pred),
        r2_score(y2_train, y2_train_pred),
    ],
    "Test R2": [
        r2_score(y1_test, y1_test_pred),
        r2_score(y2_test, y2_test_pred),
    ],
    "R2 차이(Train-Test)": [
        r2_score(y1_train, y1_train_pred) - r2_score(y1_test, y1_test_pred),
        r2_score(y2_train, y2_train_pred) - r2_score(y2_test, y2_test_pred),
    ],
    "MAE(Test)": [
        mean_absolute_error(y1_test, y1_test_pred),
        mean_absolute_error(y2_test, y2_test_pred),
    ],
    "RMSE(Test)": [
        mean_squared_error(y1_test, y1_test_pred) ** 0.5,
        mean_squared_error(y2_test, y2_test_pred) ** 0.5,
    ],
}).set_index("모델")
display(metrics.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, actual, pred, target, color in [
    (axes[0], y1_test, y1_test_pred, "Y1", COLORS["Y1"]),
    (axes[1], y2_test, y2_test_pred, "Y2", COLORS["Y2"]),
]:
    ax.scatter(actual, pred, alpha=0.55, color=color)
    lo = min(actual.min(), pred.min())
    hi = max(actual.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], "k--", label="ideal prediction")
    ax.set_xlabel(f"Actual {target}")
    ax.set_ylabel(f"Predicted {target}")
    ax.set_title(f"Actual vs Predicted: {target} Test")
    ax.legend()
plt.tight_layout()
plt.show()

**모델 결과 해석**

- Y1 Test R²는 **0.9059**로 테스트 표본에서 Y1 변동의 약 **90.6%**를 모델이 설명했다.
  Test MAE는 **2.3250 Y1 단위**, RMSE는 **3.1320 Y1 단위**다.
- Y2 Test R²는 **0.8869**로 테스트 표본에서 Y2 변동의 약 **88.7%**를 모델이 설명했다.
  Test MAE는 **2.2909 Y2 단위**, RMSE는 **3.2368 Y2 단위**다.
- R²는 예측 정확도와 같은 뜻이 아니라 목표변수 변동 중 선형모델이 설명한 비율이다.
- 두 모델 모두 RMSE가 MAE보다 크므로 일부 상대적으로 큰 오차의 영향을 받았을 가능성이 있다.
- Y1의 Train-Test R² 차이는 **0.0061**, Y2는 **-0.0071**이다. 두 분할의 설명력이 비슷하지만
  이것만으로 일반화 성능이 완전히 보장된다고 단정하지 않는다.
- 다른 변수를 고정할 때 X5 회귀계수는 Y1 **5.4073**, Y2 **5.5232**이고 X7은 각각 **20.8015**,
  **15.0672**다. 이는 각 모델 안의 조건부 선형 연관성이며 직접적인 인과효과가 아니다.
- 계수는 관찰된 설계 범위 안의 연관성을 설명하며 인과효과로 단정하지 않는다.

### 학습 기록 4: 모델 숫자를 어떻게 평가했는가?

- R² 하나만 높다고 좋은 모델이라고 결론 내리지 않고 RMSE·MAE와 Y1 표준편차를 비교했다.
- 두 테스트 RMSE(Y1 3.13, Y2 3.24)는 각 목표변수 표준편차(Y1 10.09, Y2 9.51)보다 작아 기준모델로 유용하다고 판단했다.
- 두 모델 모두 훈련·테스트 R² 차이가 작아 데이터 분할에 따른 성능 급락은 없었다.
- 하지만 실제값-예측값 그림과 잔차를 확인하니 특정 예측 구간에 오차 층이 남았다.
- 따라서 “완성된 최적 모델”이 아니라 수업 범위 안에서 만든 설명 가능한 선형 기준모델로 결론 내렸다.

## 6. 선형회귀 가정 점검

In [ ]:
residual_y1 = y1_test - y1_test_pred
residual_y2 = y2_test - y2_test_pred

fig, axes = plt.subplots(2, 3, figsize=(15, 8.2))
sns.scatterplot(x=y1_test_pred, y=residual_y1, alpha=.6, color="#264653", ax=axes[0, 0])
axes[0, 0].axhline(0, color="#E76F51", ls="--")
axes[0, 0].set(xlabel="Fitted Y1", ylabel="Residual", title="Y1 residual vs fitted")
sns.histplot(residual_y1, kde=True, color="#2A9D8F", ax=axes[0, 1])
axes[0, 1].set_title("Y1 residual distribution")
sm.qqplot(residual_y1, line="45", ax=axes[0, 2], fit=True)
axes[0, 2].set_title("Y1 residual Q-Q")

sns.scatterplot(x=y2_test_pred, y=residual_y2, alpha=.6, color="#457B9D", ax=axes[1, 0])
axes[1, 0].axhline(0, color="#E76F51", ls="--")
axes[1, 0].set(xlabel="Fitted Y2", ylabel="Residual", title="Y2 residual vs fitted")
sns.histplot(residual_y2, kde=True, color="#F4A261", ax=axes[1, 1])
axes[1, 1].set_title("Y2 residual distribution")
sm.qqplot(residual_y2, line="45", ax=axes[1, 2], fit=True)
axes[1, 2].set_title("Y2 residual Q-Q")

plt.tight_layout()
plt.show()

for target, actual, pred, residual in [
    ("Y1", y1_test, y1_test_pred, residual_y1),
    ("Y2", y2_test, y2_test_pred, residual_y2),
]:
    print(f"[{target}] residual mean: {residual.mean():.4f}")
    print(f"[{target}] residual skewness: {residual.skew():.4f}")
    print(
        f"[{target}] corr(fitted, residual): "
        f"{pd.Series(pred, index=actual.index).corr(residual):.4f}"
    )
    print(f"[{target}] min residual: {residual.min():.4f}")
    print(f"[{target}] max residual: {residual.max():.4f}")
    print(f"[{target}] largest absolute residual: {residual.abs().max():.4f}")

**가정 해석**

- **실제값-예측값:** 두 테스트 산점도는 대체로 기준선 주변에 놓이지만, Y1·Y2의 높은 실제값 구간에서
  기준선과 비교적 멀리 떨어진 점들이 관찰된다.
- **잔차 평균:** Y1은 **0.3002**, Y2는 **0.4175**로 0에 가깝지만 완전히 0은 아니다.
- **선형성:** 두 모델 모두 잔차가 0선을 중심으로 퍼지지만 일부 예측 구간에서 층과 굴곡이 보여
  완전한 선형관계가 충족된다고 단정하기 어렵다.
- **등분산성:** 예측값 약 25 이상 구간에서 잔차 폭이 더 넓어지는 모습이 있어 분산이 모든 구간에서
  일정하다고 보기 어렵다.
- **정규성:** Y1 잔차 왜도는 **0.083**으로 대칭에 가깝지만 Y2 잔차 왜도는 **0.883**으로 오른쪽 치우침이
  더 크다. 두 Q-Q plot 모두 중앙부는 직선과 비슷하지만 꼬리 부분은 벗어난다.
- **큰 오차:** 최대 절대잔차는 Y1 약 **8.262**, Y2 약 **11.671**로, 특히 Y2에 상대적으로 큰 양의 잔차가
  관찰된다. RMSE가 MAE보다 큰 결과와도 일치한다.
- **독립성:** 각 행은 서로 다른 시뮬레이션 설계 조합이므로 시간 순서 자료보다는 독립에 가깝다. 다만 같은 기본 형상을
  반복한 설계가 포함되어 완전한 독립을 보장한다고 단정하지 않는다.
- **중복 설명변수:** X1·X4를 제외하여 강한 형상 중복을 줄였다. X2·X3·X5에도 관계가 남아 있으므로 개별 계수보다
  전체 설명력과 방향을 중심으로 해석한다.

따라서 두 모델은 수업 범위 안의 **기초 선형 기준모델**로 의미가 있지만, 잔차의 층 구조·구간별 분산 차이·
꼬리 이탈 때문에 선형성·정규성·등분산성이 완전히 충족된다고 말할 수 없다.

## 7. 결론과 비즈니스/설계 인사이트

1. 이 데이터에서는 전체 높이와 표면적 등 건물 형상 관련 변수가 Y1과 Y2에 비교적 강한 선형관계를 보였다.
   다만 상관관계와 회귀계수는 연관성을 나타내며 인과관계로 해석할 수 없다.
2. X7과 목표변수 사이에서 관찰된 관계는 창호 면적과 부하가 함께 변하는 경향을 보여준다. 현재 데이터와
   선형회귀 결과만으로 열손실이나 일사 유입 같은 원인을 확정할 수는 없다.
3. X6은 범주형 방향 코드로 그룹별 분포를 비교했다. 네 그룹의 중심 차이는 그룹 내부 퍼짐보다 작게 관찰되었지만,
   방향의 일반적 효과가 없다고 단정하지 않는다.
4. X8은 코드형 범주변수이므로 연속형 숫자로 선형회귀에 넣지 않았다. 범주 효과를 더 분석하려면 별도의 범주형
   처리 방식이 필요하지만 이번 과제의 단순한 기초통계·선형회귀 범위에서는 제외했다.
5. 초기 설계안 비교에서 X2·X3·X5·X7을 공통 입력으로 사용해 Y1·Y2 부하의 선형 기준값을 각각 만들 수 있다.
   다만 큰 잔차가 관찰되는 설계군은 상세 분석이 필요하다.
6. 시뮬레이션 데이터이며 기후·재료·운영시간 변수가 없다. 결과를 실제 건물 전체로 바로 일반화할 수 없다.

